In [1]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np
import os
from pandas_datareader import data as pdr

In [2]:
# MENGUMPULKAN DATA KUOTASI OPSI NVIDIA
folder_path = r"D:\SKRIPSI"

files_to_merge = (
    [f"2022_{m:02d}.txt" for m in range(4, 13)] +
    [f"2023_{m:02d}.txt" for m in range(1, 13)]
)

df = pd.concat(
    [pd.read_csv(os.path.join(folder_path, f), low_memory=False) 
     for f in files_to_merge],
    ignore_index=True
)

In [3]:
# MENYIMPAN DATA KUOTASI OPSI YANG MASIH MENTAH
df.to_csv("Data_Kuotasi_Opsi_NVDA_Mentah.csv", index=False)

In [4]:
# MEMBERSIHKAN FORMAT NAMA KOLOM DAN TIPE DATA
df.columns = (
    df.columns
      .str.strip()
      .str.replace('[', '', regex=False)
      .str.replace(']', '', regex=False)
)

for i in [8,9,10,11,12,15,17,18,20,21,23,24,25,26,27,28]:
    col_name = df.columns[i]
    df[col_name] = df[col_name].astype(str).str.strip()
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")

In [5]:
# MEMBERSIHKAN DATA YANG TIDAK VALID
df["ID_OPTION"] = (
    df["EXPIRE_UNIX"].astype(str) + "_" +
    df["STRIKE"].astype(str)
)

df["invalid_bid"] = (df["P_BID"].isna())
df["invalid_ask"] = (df["P_ASK"] == 0) | (df["P_ASK"].isna())
df["invalid_bid_ask_spread"] = df["P_BID"] > df["P_ASK"]
df["invalid_underlying"] = (df["UNDERLYING_LAST"] == 0) | (df["UNDERLYING_LAST"].isna())
df["invalid_strike"] = (df["STRIKE"] == 0) | (df["STRIKE"].isna())
df["invalid_dte"] = (df["DTE"].isna())

invalid_mask = (
    df["invalid_bid"] |
    df["invalid_ask"] |
    df["invalid_bid_ask_spread"] |
    df["invalid_underlying"] |
    df["invalid_strike"] |
    df["invalid_dte"]
)

invalid_contracts = df.loc[invalid_mask, "ID_OPTION"].unique()

df_clean = (
    df[~df["ID_OPTION"].isin(invalid_contracts)]
      .loc[lambda x: x["DTE"] <= 730]
      .copy()
)
df_clean.reset_index(drop=True, inplace=True)

df_clean = df_clean[
    df_clean.groupby("ID_OPTION")["ID_OPTION"].transform('count') >= 5
].copy()

df_clean.reset_index(drop=True, inplace=True)

In [6]:
# MENYELEKSI KOLOM YANG DIBUTUHKAN
columns_to_keep = [
    'QUOTE_DATE',
    'ID_OPTION',
    'UNDERLYING_LAST',
    'EXPIRE_DATE',
    'DTE',
    'STRIKE',
    'P_BID',
    'P_ASK'
]
df_clean = df_clean[columns_to_keep]

In [7]:
# MENYIMPAN DATA KUOTASI OPSI PUT SAHAM NVIDIA CLEANING
df_clean.to_csv("Data_Kuotasi_Opsi_Put_NVDA_Cleaning.csv", index=False)

In [8]:
# MENGUMPULKAN DATA HARGA PENUTUPAN HARIAN SAHAM NVIDIA
folder_path = r"D:\SKRIPSI"

files_to_merge = (
    [f"2021_{m:02d}.txt" for m in range(11, 13)] +
    [f"2022_{m:02d}.txt" for m in range(1, 13)] +
    [f"2023_{m:02d}.txt" for m in range(1, 13)]
)

df = pd.concat(
    [pd.read_csv(os.path.join(folder_path, f), low_memory=False) 
     for f in files_to_merge],
    ignore_index=True
)

In [9]:
# MEMBERSIHKAN FORMAT NAMA KOLOM DAN TIPE DATA
df.columns = (
    df.columns
      .str.strip()
      .str.replace('[', '', regex=False)
      .str.replace(']', '', regex=False)
)

for i in [8,9,10,11,12,15,17,18,20,21,23,24,25,26,27,28]:
    col_name = df.columns[i]
    df[col_name] = df[col_name].astype(str).str.strip()
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")

In [10]:
# MEMBERSIHKAN DUPLIKAT HARGA PENUTUPAN HARIAN SAHAM PADA HARI YANG SAMA
df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'])
harga_saham = df[['QUOTE_DATE', 'UNDERLYING_LAST']].drop_duplicates()
harga_saham = harga_saham.sort_values("QUOTE_DATE")

In [11]:
# MENYELEKSI PERIODE DATA
harga_saham = harga_saham[
    (harga_saham["QUOTE_DATE"] >= "2021-11-23") &
    (harga_saham["QUOTE_DATE"] <= "2023-12-29")
]

In [12]:
# MENIMPAN DATA HARGA PENUTUPAN HARIAN SAHAM 
harga_saham.to_excel("Data_Harga_Penutupan_Harian_Saham.xlsx", index=False)

In [13]:
# MENGUMPULKAN DATA TINGKAT SUKU BUNGA
start = "2022-03-28"
end = "2023-12-29"

series = {
    "Treasury_1_Bulan": "DGS1MO",
    "Treasury_3_Bulan": "DGS3MO",
    "Treasury_6_Bulan": "DGS6MO",
    "Treasury_1_Tahun": "DGS1",
    "Treasury_2_Tahun": "DGS2"
}

df = pdr.DataReader(list(series.values()), "fred", start, end)
df = df.rename(columns={v: k for k, v in series.items()})

df = df.ffill()

df = df.reset_index()
df = df.rename(columns={"DATE": "QUOTE_DATE", "index": "QUOTE_DATE"})

In [14]:
# MENYIMPAN DATA TINGKAT SUKU BUNGA
df.to_excel("Data_Tingkat_Suku_Bunga.xlsx", index=False)